# HSTU_CANONICAL_v1 — Finalizer v3

This version fixes both earlier launcher issues:

- Triton `@jit` functions are written to a **real `.py` file** before import,
  so Triton 3.6 can inspect their source.
- Core/large checkpoints are auto-discovered anywhere under `MyDrive`,
  preferring `best.pt`, then `latest.pt`, then `last.pt`.

No HSTU training or quality logic is changed.

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Switch Colab runtime to GPU."

try:
    import triton
    print("Triton:", triton.__version__)
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton
    print("Triton installed:", triton.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
# Build a REAL Python module from the canonical notebook's definition cells.
# This is important: Triton 3.6 rejects @jit functions created via exec()
# from synthetic filenames because inspect.getsourcelines() cannot find them.
import urllib.request, json, importlib.util, sys
from pathlib import Path

CANONICAL_URL = "https://raw.githubusercontent.com/hanialshater/Sparsewalker-/main/experiments/hstu_reproduction/HSTU_CANONICAL_v1_Finalize_Colab.ipynb"
RUNTIME_PY = Path("/content/hstu_canonical_runtime.py")

with urllib.request.urlopen(CANONICAL_URL) as r:
    canonical = json.load(r)

chunks = []
for i, cell in enumerate(canonical["cells"]):
    if cell.get("cell_type") != "code":
        continue
    src = "".join(cell.get("source", []))

    # Skip only notebook execution/display cells. Keep all definitions,
    # including the Triton kernel source.
    if "stamp, summary_df, latency_df = finalize()" in src:
        continue
    if 'print("HSTU_CANONICAL_v1:"' in src:
        continue
    if "latency_df.pivot_table" in src:
        continue

    chunks.append(f"\n# ---- canonical notebook cell {i} ----\n")
    chunks.append(src)
    chunks.append("\n")

RUNTIME_PY.write_text("".join(chunks))
print("Wrote real source module:", RUNTIME_PY, "bytes:", RUNTIME_PY.stat().st_size)

spec = importlib.util.spec_from_file_location("hstu_canonical_runtime", RUNTIME_PY)
hstu = importlib.util.module_from_spec(spec)
sys.modules["hstu_canonical_runtime"] = hstu
spec.loader.exec_module(hstu)

assert callable(hstu.finalize)
print("Imported:", hstu.__file__)
print("finalize(): OK")
print("Triton available in module:", hstu.TRITON_AVAILABLE)

In [ ]:
# Patch ONLY checkpoint resolution.
from pathlib import Path

_resolved = {}

def discover_checkpoint(variant):
    if variant in _resolved:
        return _resolved[variant]

    mydrive = Path("/content/drive/MyDrive")
    folder_name = f"hstu_{variant}_seed{hstu.SEED}"
    found = []

    # Prefer expected root, but do not require it.
    expected = Path("/content/drive/MyDrive/hstu_pure_pytorch_ml1m") / folder_name
    for rank, name in enumerate(("best.pt", "latest.pt", "last.pt")):
        p = expected / name
        if p.exists():
            found.append((rank, p))

    # Recursive fallback anywhere in MyDrive.
    seen = {str(p) for _, p in found}
    for rank, name in enumerate(("best.pt", "latest.pt", "last.pt")):
        for p in mydrive.rglob(name):
            if str(p) in seen:
                continue
            if p.parent.name == folder_name:
                found.append((rank, p))
                seen.add(str(p))

    if not found:
        nearby = [
            str(p) for p in mydrive.rglob("*.pt")
            if "hstu" in str(p).lower()
        ][:50]
        raise FileNotFoundError(
            f"No checkpoint found for {folder_name}.\n"
            "HSTU checkpoints visible in this MyDrive:\n"
            + "\n".join(nearby)
        )

    # best.pt > latest.pt > last.pt; newest file breaks ties.
    found.sort(key=lambda rp: (rp[0], -rp[1].stat().st_mtime))
    chosen = found[0][1]
    _resolved[variant] = chosen

    print("RESOLVED CHECKPOINT:", {
        "variant": variant,
        "path": str(chosen),
        "kind": chosen.name,
    })
    return chosen

# load_canonical_model() looks up checkpoint_path in its module globals,
# so replacing this single function patches both loading and stamp metadata.
hstu.checkpoint_path = discover_checkpoint

# Preflight before any dataset/parity/latency work.
for variant in hstu.VARIANTS:
    discover_checkpoint(variant)

In [ ]:
# Run complete stamp suite.
stamp, summary_df, latency_df = hstu.finalize()
display(summary_df)
display(latency_df)

In [ ]:
print("HSTU_CANONICAL_v1:", stamp["status"])
print("Stamp file:", hstu.STAMP_PATH)

if len(latency_df):
    display(
        latency_df.pivot_table(
            index=["variant", "batch", "history"],
            columns="backend",
            values="p50_ms",
        ).reset_index()
    )